# 12 — SCD Tipo 3 (Histórico Limitado) — Spark SQL

`DimCustomerSCD3` com `CurrentCity` / `PreviousCity` / `CityChangedOn`.

**Técnica Spark SQL:** carga via `INSERT INTO SELECT`, updates via `spark.sql("UPDATE ...")`.
Sem DeltaTable API.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR
from datetime import date

spark = get_spark("NorthwindDW SQL - 12 SCD3")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 01:00:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 13}


In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.DimCustomerSCD3 (
        CustomerID STRING, CompanyName STRING, ContactName STRING,
        CurrentCity STRING, PreviousCity STRING, CityChangedOn DATE,
        CurrentCountry STRING, PreviousCountry STRING, CountryChangedOn DATE,
        LoadTimestamp TIMESTAMP
    ) USING DELTA
""")

spark.sql("DELETE FROM gold.DimCustomerSCD3")
spark.sql("""
    INSERT INTO gold.DimCustomerSCD3
    SELECT CustomerID, CompanyName, ContactName,
           City AS CurrentCity, NULL AS PreviousCity, NULL AS CityChangedOn,
           Country AS CurrentCountry, NULL AS PreviousCountry, NULL AS CountryChangedOn,
           current_timestamp() AS LoadTimestamp
    FROM bronze.customers
""")

n = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomerSCD3").collect()[0]["n"]
print(f"DimCustomerSCD3 carga inicial: {n} clientes")
assert n == 91
print("Carga inicial OK")

DimCustomerSCD3 carga inicial: 91 clientes
Carga inicial OK


In [4]:
def process_customer_scd3():
    """Detecta mudanças de City/Country e aplica lógica SCD3 via MERGE INTO."""
    today = str(date.today())

    # ── Cidade ────────────────────────────────────────────────────────────────
    city_changed_n = spark.sql("""
        SELECT COUNT(*) AS n
        FROM bronze.customers b
        JOIN gold.DimCustomerSCD3 s ON b.CustomerID = s.CustomerID
        WHERE COALESCE(b.City, '') <> COALESCE(s.CurrentCity, '')
    """).collect()[0]['n']

    if city_changed_n > 0:
        # MERGE INTO: fonte tem a nova cidade, alvo recebe PreviousCity=CurrentCity
        spark.sql(f"""
            MERGE INTO gold.DimCustomerSCD3 AS tgt
            USING (
                SELECT b.CustomerID, b.City AS NewCity
                FROM bronze.customers b
                JOIN gold.DimCustomerSCD3 s ON b.CustomerID = s.CustomerID
                WHERE COALESCE(b.City, '') <> COALESCE(s.CurrentCity, '')
            ) AS src
            ON tgt.CustomerID = src.CustomerID
            WHEN MATCHED THEN UPDATE SET
                tgt.PreviousCity  = tgt.CurrentCity,
                tgt.CityChangedOn = CAST('{today}' AS DATE),
                tgt.CurrentCity   = src.NewCity
        """)
        print(f"SCD3: {city_changed_n} mudanças de cidade aplicadas")
    else:
        print("SCD3: nenhuma mudança de cidade detectada")

    # ── País ──────────────────────────────────────────────────────────────────
    country_changed_n = spark.sql("""
        SELECT COUNT(*) AS n
        FROM bronze.customers b
        JOIN gold.DimCustomerSCD3 s ON b.CustomerID = s.CustomerID
        WHERE COALESCE(b.Country, '') <> COALESCE(s.CurrentCountry, '')
    """).collect()[0]['n']

    if country_changed_n > 0:
        spark.sql(f"""
            MERGE INTO gold.DimCustomerSCD3 AS tgt
            USING (
                SELECT b.CustomerID, b.Country AS NewCountry
                FROM bronze.customers b
                JOIN gold.DimCustomerSCD3 s ON b.CustomerID = s.CustomerID
                WHERE COALESCE(b.Country, '') <> COALESCE(s.CurrentCountry, '')
            ) AS src
            ON tgt.CustomerID = src.CustomerID
            WHEN MATCHED THEN UPDATE SET
                tgt.PreviousCountry  = tgt.CurrentCountry,
                tgt.CountryChangedOn = CAST('{today}' AS DATE),
                tgt.CurrentCountry   = src.NewCountry
        """)
        print(f"SCD3: {country_changed_n} mudanças de país aplicadas")
    else:
        print("SCD3: nenhuma mudança de país detectada")


In [5]:
print("--- Estado ANTES ---")
spark.sql("""
    SELECT CustomerID, CurrentCity, PreviousCity, CityChangedOn
    FROM gold.DimCustomerSCD3 WHERE CustomerID IN ('ALFKI', 'ANATR', 'BOLID')
""").show()

spark.sql("UPDATE bronze.customers SET City = 'Lyon'     WHERE CustomerID = 'ALFKI'")
spark.sql("UPDATE bronze.customers SET City = 'Madrid'   WHERE CustomerID = 'ANATR'")
spark.sql("UPDATE bronze.customers SET City = 'Valencia' WHERE CustomerID = 'BOLID'")

process_customer_scd3()

print("--- Estado DEPOIS ---")
spark.sql("""
    SELECT CustomerID, CurrentCity AS CidadeAtual,
           PreviousCity AS CidadeAnterior, CityChangedOn AS DataMudanca
    FROM gold.DimCustomerSCD3 WHERE CustomerID IN ('ALFKI', 'ANATR', 'BOLID')
""").show()

spark.sql("UPDATE bronze.customers SET City = 'Berlin'      WHERE CustomerID = 'ALFKI'")
spark.sql("UPDATE bronze.customers SET City = 'México D.F.' WHERE CustomerID = 'ANATR'")
spark.sql("UPDATE bronze.customers SET City = 'Madrid'      WHERE CustomerID = 'BOLID'")
print("Bronze restaurado")

--- Estado ANTES ---


+----------+-----------+------------+-------------+
|CustomerID|CurrentCity|PreviousCity|CityChangedOn|
+----------+-----------+------------+-------------+
|     ALFKI|     Berlin|        NULL|         NULL|
|     ANATR|México D.F.|        NULL|         NULL|
|     BOLID|     Madrid|        NULL|         NULL|
+----------+-----------+------------+-------------+



SCD3: 3 mudanças de cidade aplicadas


SCD3: nenhuma mudança de país detectada
--- Estado DEPOIS ---


+----------+-----------+--------------+-----------+
|CustomerID|CidadeAtual|CidadeAnterior|DataMudanca|
+----------+-----------+--------------+-----------+
|     ALFKI|       Lyon|        Berlin| 2026-03-29|
|     ANATR|     Madrid|   México D.F.| 2026-03-29|
|     BOLID|   Valencia|        Madrid| 2026-03-29|
+----------+-----------+--------------+-----------+



Bronze restaurado


In [6]:
spark.sql("""
    SELECT CustomerID, CompanyName,
           CurrentCity AS CidadeAtual, PreviousCity AS CidadeAnterior, CityChangedOn AS MudouEm
    FROM gold.DimCustomerSCD3 WHERE PreviousCity IS NOT NULL ORDER BY CityChangedOn DESC
""").show()

+----------+--------------------+-----------+--------------+----------+
|CustomerID|         CompanyName|CidadeAtual|CidadeAnterior|   MudouEm|
+----------+--------------------+-----------+--------------+----------+
|     ALFKI| Alfreds Futterkiste|       Lyon|        Berlin|2026-03-29|
|     ANATR|Ana Trujillo Empa...|     Madrid|   México D.F.|2026-03-29|
|     BOLID|Bólido Comidas pr...|   Valencia|        Madrid|2026-03-29|
+----------+--------------------+-----------+--------------+----------+

